# Building a custom robot config for cuRobo

cuRobo only ships a handful of example robot configs by name (`franka.yml`, `ur10e.yml`, `dual_ur10e.yml`, `unitree_g1.yml`, ...). If your robot isn't one of these, you need to build your own config from a URDF - including fitting collision spheres yourself, since cuRobo represents robot links as spheres for fast GPU collision checking.

cuRobo v2 ships a first-party `RobotBuilder` (`curobo.robot_builder`) for this, replacing the old external "bubblify" tool from cuRobo v1. This notebook walks through the whole pipeline using cuRobo's bundled `ur10e.urdf` as a concrete, runnable example - the same steps apply to any URDF of your own.

**Every step below was actually run against a real GPU while writing this notebook** - including two real gotchas in `RobotBuilder` that aren't documented upstream (see the cells marked ⚠️), and a genuine diagnostic walkthrough at the end where planning fails for real and we work out why.

## 1. Locate your URDF and mesh assets

`RobotBuilder` resolves paths relative to cuRobo's own installed assets directory (`curobo.content.get_assets_path()`). If your URDF and meshes live *inside* that directory tree, the saved config stores portable relative paths; otherwise it falls back to absolute paths (which still work, just aren't portable to another machine).

For a robot you're adding to your own cuRobo checkout, the natural place to put it is `<curobo>/curobo/content/assets/robot/<your_robot>/`, alongside the existing bundled robots (e.g. `ur_description/`, `franka_description/`).

**If your robot is one that [`airo-models`](https://github.com/airo-ugent/airo-models) already ships** (UR3e, UR5e, Robotiq 2F-85, ...), use that URDF directly instead of hand-rolling your own - it already has correctly-resolving meshes, unlike a from-scratch URDF where you have to get mesh paths right yourself (see the next cell for exactly why that matters).

In [ ]:
from pathlib import Path

from curobo.content import get_assets_path

# Substitute your own URDF + asset directory here. This example reuses cuRobo's bundled UR10e
# so the notebook is runnable out of the box.
urdf_path = Path(get_assets_path()) / "robot/ur_description/ur10e.urdf"
asset_path = Path(get_assets_path()) / "robot/ur_description"
assert urdf_path.exists(), f"URDF not found: {urdf_path}"

# If your URDF references mesh files (e.g. <mesh filename="visual/forearm.obj"/>), those files must
# actually exist under asset_path - RobotBuilder will fail to load meshes otherwise. This bit us
# directly: a UR3e URDF placed under ur_description/ referenced meshes that were never added
# alongside it (only ur5e/ and ur10e/ mesh subdirectories exist there), so building it failed.
# Double check with e.g. `grep -o 'filename="[^"]*"' your.urdf` before proceeding.

### Using an `airo-models` URDF instead

`airo_models.get_urdf_path(name)` returns the absolute path to a bundled URDF (e.g. `"ur3e"`, `"ur5e"`, `"robotiq_2f_85"`). Unlike cuRobo's own robots (which keep meshes in a shared `meshes/<robot>/visual/...` subdirectory), `airo-models` keeps each URDF next to its own `visual/`/`collision/` mesh folders - so the asset path `RobotBuilder` needs is simply the URDF's own parent directory. This was verified end-to-end (mesh loading, sphere fitting, self-collision matrix, and actual planning with `SingleArmCuroboPlanner`) against `airo_models`' bundled UR3e:

In [ ]:
import os

import airo_models

airo_models_urdf_path = airo_models.get_urdf_path("ur3e")
airo_models_asset_path = os.path.dirname(airo_models_urdf_path)  # URDF's own directory, not a "meshes/" subfolder

print(airo_models_urdf_path)

# Note: airo_models.AIRO_MODEL_NAMES only lists a subset (arms); get_urdf_path also accepts grippers,
# cameras, mobile-platform parts, and mounting plates - check airo_models/files.py for the full list.

# To actually use it instead of the ur10e.urdf example below, uncomment:
# urdf_path, asset_path = airo_models_urdf_path, airo_models_asset_path

## 2. Create the builder and fit collision spheres

`fit_collision_spheres` approximates each link's mesh with a set of spheres. `sphere_density` controls how many spheres per link (practical range 0.1-10.0); `compute_metrics=True` reports fit quality (`coverage` close to 1.0 is good, high `protrusion` means spheres bulge outside the mesh).

In [ ]:
from curobo.robot_builder import RobotBuilder

builder = RobotBuilder(str(urdf_path), str(asset_path), tool_frames=["tool0"])
builder.fit_collision_spheres(sphere_density=1.0, compute_metrics=True)

print(f"Fitted {builder.num_spheres} spheres across {len(builder.collision_link_names)} links")
for link, metrics in builder.link_metrics.items():
    print(f"  {link}: {metrics.num_spheres} spheres, coverage={metrics.coverage:.2f}")

### Optional: inspect the fit interactively

`builder.visualize()` opens an interactive Viser viewer so you can see the fitted spheres against the robot. **It blocks the cell** until `timeout_sec` elapses (or forever if you don't pass one) - open the printed URL in your browser, then either wait for the timeout or interrupt the cell when you're done looking.

⚠️ **Gotcha:** `show_meshes=True` is unreliable - don't use it to judge sphere fit quality. Each link's mesh is added as a flat, static Viser node positioned at that mesh's local `<visual>` origin *from the URDF*, without ever being passed through the robot's actual kinematic chain (confirmed by reading `RobotBuilder.visualize()`'s and `ViserVisualizer.add_mesh()`'s source). Only the base link's mesh ends up near its correct world position by coincidence; every other link's mesh appears frozen at the wrong place, visibly detached from the correctly-posed sphere set and skeleton (which *do* go through forward kinematics). Leave `show_meshes=False` (the default) and rely on the `coverage`/`protrusion` metrics from step 2, or `RobotDebugger`, instead.

In [ ]:
# builder.visualize(timeout_sec=30)  # uncomment to try it - leave show_meshes at its default (False)

If a specific link's fit is bad, refit just that link instead of the whole robot:
```python
builder.refit_link_spheres("wrist_3_link", sphere_density=2.0)
```
or clip spheres away from one side of a link (e.g. to avoid the base link's spheres poking below the mounting plate):
```python
builder.fit_collision_spheres(clip_links={"base_link_inertia": ("z", 0.0)})
```

## 3. Compute the self-collision ignore matrix

This figures out which link pairs should never be checked against each other: neighboring links (always ignored), links that collide at the *current default joint configuration* (treated as false positives), and - optionally - pairs sampled to never collide across many random configurations (pruned for efficiency).

⚠️ **Gotcha:** the "collide at default configuration" check uses whatever the robot's current default joint position is *at this point in the pipeline* - which is all-zeros unless you've loaded an existing config with `RobotBuilder.from_config(...)`. If you plan to use a different "home"/retract pose later, false positives at *that* pose won't be caught here - verify separately with `RobotDebugger` in step 5.

In [ ]:
builder.compute_collision_matrix(num_samples=1000)
print(f"{len(builder.collision_matrix)} links have ignore entries")

## 4. Build and save

⚠️ **Gotcha:** `builder.save(...)` writes two internal loader flags (`load_collision_spheres`, `num_envs`) into the YAML's `kinematics:` block. When you later load that same file the normal way (e.g. `MotionPlannerCfg.create(robot="my_robot.yml")`, which is what `SingleArmCuroboPlanner` does), cuRobo's loader passes those same two flags explicitly *and* unpacks the YAML's own `kinematics:` dict as keyword arguments - causing a `TypeError: got multiple values for keyword argument 'load_collision_spheres'`. Strip both keys after saving; the file works fine without them.

In [ ]:
import yaml

config = builder.build()
output_path = "/tmp/ur10e_custom.yml"
builder.save(config, output_path)

# Work around the load_collision_spheres/num_envs bug described above.
with open(output_path) as f:
    data = yaml.safe_load(f)
data["kinematics"].pop("load_collision_spheres", None)
data["kinematics"].pop("num_envs", None)
with open(output_path, "w") as f:
    yaml.safe_dump(data, f)

## 5. Verify self-collision at your chosen default pose

`RobotDebugger` checks *self*-collision only (not collision with the world/scene). Since `compute_collision_matrix()` (step 3) only auto-suppressed false positives at the all-zeros pose, explicitly check whichever pose you actually intend to use as "home".

In [ ]:
from curobo.robot_builder import RobotDebugger

debugger = RobotDebugger(output_path)
result = debugger.check_default_joint_configuration_collision()
print("Self-collision at default pose:", result["has_collision"])

# To check a specific non-default "home" pose instead:
# result = debugger.check_collision_at_config([0.0, -2.2, 1.9, -1.383, -1.57, 0.0])

## 6. Use it with `SingleArmCuroboPlanner`

This is the real end-to-end test - self-collision passing (step 5) does **not** guarantee the robot won't collide with *your scene*. In fact, with this notebook's default `sphere_density=1.0` fit for the UR10e, planning against cuRobo's bundled `collision_test.yml` scene genuinely fails below, even though self-collision was clean - this is real, reproducible behavior, not a hypothetical.

In [ ]:
from airo_planner import NoPathFoundError
from airo_planner.curobo.single_arm_curobo_planner import SingleArmCuroboPlanner

planner = SingleArmCuroboPlanner(output_path, "collision_test.yml")

q_start = planner.motion_planner.default_joint_state.position.cpu().numpy()
q_goal = q_start.copy()
q_goal[0] += 0.5
q_goal[1] += 0.3

try:
    trajectory = planner.plan_to_joint_configuration(q_start, q_goal)
    print(f"Planned {len(trajectory.path.positions)} waypoints")
except NoPathFoundError:
    print("Planning failed against collision_test.yml - diagnosing below.")

### Diagnosing a failure that self-collision didn't predict

Since `RobotDebugger` (step 5) already ruled out self-collision, the next question is whether this is a *world*-collision (fitted spheres bulging into the scene's obstacles) or something else. Isolate it by re-running the identical plan with **no** world obstacles at all - if that succeeds, the fitted spheres themselves are fine in isolation, and the problem is specifically their size/placement relative to this particular scene.

In [ ]:
import torch
from curobo.motion_planner import MotionPlanner, MotionPlannerCfg
from curobo.types import JointState

empty_world_config = MotionPlannerCfg.create(robot=output_path, use_cuda_graph=False)  # no scene_model
empty_world_planner = MotionPlanner(empty_world_config)
empty_world_planner.warmup(num_warmup_iterations=1)

start_js = JointState.from_position(
    torch.tensor(q_start, dtype=torch.float32)[None, :].cuda(), joint_names=empty_world_planner.joint_names
)
goal_js = JointState.from_position(
    torch.tensor(q_goal, dtype=torch.float32)[None, :].cuda(), joint_names=empty_world_planner.joint_names
)
result = empty_world_planner.plan_cspace(goal_js, start_js)
print("Plan succeeds with an empty world:", result is not None and bool(result.success.any()))

This confirms it: the fitted spheres are collision-free with an empty world, so the failure above is specifically about this sphere fit intersecting `collision_test.yml`'s obstacles at the tested poses - a scene/sphere-margin issue, not a bug in the config or the planner.

**This is expected, not a dead end** - it's exactly why step 6 matters: auto-fit spheres are a conservative approximation and won't always match the tighter margins of a hand-tuned config like the bundled `ur10e.yml`. From here, the fix is iterative and scene-specific:
- Test against *your actual* scene, not an unrelated bundled one like `collision_test.yml` (which was tuned around a Franka Panda's geometry, not a UR10e's).
- If it's still too conservative for your scene, try lowering `sphere_density` or `clip_links` for the specific link that's closest to the obstacle in step 2 (use the per-link `coverage`/`protrusion` metrics, or `RobotDebugger`, to judge fit quality - **not** `builder.visualize(show_meshes=True)`, which is unreliable, see step 2), then repeat steps 3-6.
- Re-run this section after any refit - there's no shortcut to re-verifying against your real scene.